In [17]:
import sys; sys.path.append('..')
import numpy as np, pandas as pd
from skimage.transform import resize
from sklearn.model_selection import train_test_split
from tqdm import tqdm
from src.utils import CLASS_NAMES, set_seed

SEED = 42
set_seed(SEED)
tqdm.pandas()

# ── 구버전 pandas pickle 호환용 shim ──────────────────────
import types
import pandas as pd
from pandas.core.indexes.base import Index, _new_Index

def _shim(name, **attrs):
    m = types.ModuleType(name)
    for k, v in attrs.items():
        setattr(m, k, v)
    sys.modules[name] = m

_shim('pandas.indexes')
_shim('pandas.indexes.base', _new_Index=_new_Index, Index=Index)
_shim('pandas.indexes.numeric', Int64Index=Index, Float64Index=Index)
_shim('pandas.indexes.range', RangeIndex=pd.RangeIndex)
_shim('pandas.indexes.multi', MultiIndex=pd.MultiIndex)

In [18]:
import pickle

with open('../data/LSWMD.pkl', 'rb') as f:
    df = pickle.load(f, encoding='latin1')

print(f"원본: {len(df)}")
print(df.columns.tolist())

df = df.drop(['waferIndex', 'dieSize', 'lotName'], axis=1)
df['failureType'] = df['failureType'].apply(lambda x: x[0][0] if len(x) > 0 else 'none')

print(df['failureType'].value_counts())

원본: 811457
['waferMap', 'dieSize', 'lotName', 'waferIndex', 'trianTestLabel', 'failureType']
failureType
none         785938
Edge-Ring      9680
Edge-Loc       5189
Center         4294
Loc            3593
Scratch        1193
Random          866
Donut           555
Near-full       149
Name: count, dtype: int64


In [22]:
df_failure = df[df['failureType'] != 'none'].copy()
df_none = df[df['failureType'] == 'none'].sample(n=5000, random_state=SEED).copy()
df_reduced = pd.concat([df_failure, df_none]).reset_index(drop=True)
print(f"{len(df_reduced)}장 (불량 {len(df_failure)}, 정상 5000)")


def preprocess_wafer_map(wm, target_size=(64, 64)):
    # order=0 (Nearest) → 0/1/2 정수 라벨이 중간값으로 뭉개지지 않음
    return resize(wm, target_size, order=0,
                  preserve_range=True, anti_aliasing=False).astype(np.uint8)


resized = df_reduced['waferMap'].progress_apply(preprocess_wafer_map).tolist()
X = np.array(resized, dtype=np.uint8)[:, np.newaxis, :, :]   # (N, 1, 64, 64)

mapping = {c: i for i, c in enumerate(CLASS_NAMES)}
y = df_reduced['failureType'].map(mapping).values.astype(np.int64)

print(X.shape, X.dtype, f"{X.nbytes/1024/1024:.1f} MB")


del df, df_failure, df_none, resized
import gc; gc.collect()

30519장 (불량 25519, 정상 5000)


100%|██████████| 30519/30519 [00:00<00:00, 32091.03it/s]


(30519, 1, 64, 64) uint8 119.2 MB


5047

In [25]:
del df, df_failure, df_none, resized
import gc; gc.collect()

NameError: name 'df' is not defined

In [26]:
# Test 20% 먼저 분리
X_temp, X_test, y_temp, y_test = train_test_split(
    X, y, test_size=0.2, random_state=SEED, stratify=y)

# 남은 80%에서 Valid 분리 (0.25 × 0.8 = 전체의 20%)
X_train, X_valid, y_train, y_valid = train_test_split(
    X_temp, y_temp, test_size=0.25, random_state=SEED, stratify=y_temp)

dist = pd.DataFrame({
    'train': np.bincount(y_train, minlength=9),
    'valid': np.bincount(y_valid, minlength=9),
    'test':  np.bincount(y_test,  minlength=9),
}, index=CLASS_NAMES)
dist['total'] = dist.sum(axis=1)
print(dist.sort_values('total', ascending=False))

           train  valid  test  total
Edge-Ring   5808   1936  1936   9680
Edge-Loc    3113   1038  1038   5189
none        3000   1000  1000   5000
Center      2576    859   859   4294
Loc         2156    718   719   3593
Scratch      716    239   238   1193
Random       520    173   173    866
Donut        333    111   111    555
Near-full     89     30    30    149


In [27]:
import os
os.makedirs('../data/cache', exist_ok=True)
np.savez_compressed('../data/cache/split.npz',
    X_train=X_train, y_train=y_train,
    X_valid=X_valid, y_valid=y_valid,
    X_test=X_test,   y_test=y_test)
print("saved:", os.path.getsize('../data/cache/split.npz')/1024/1024, "MB")

dist.to_csv('../results/class_distribution.csv', encoding='utf-8-sig')

saved: 7.770750045776367 MB
